In [0]:
%sql
describe workspace.pj_sales.tb_sales_silver

In [0]:
%sql
create table if not exists 
  workspace.pj_sales.tb_dim_car_gold (
     _sk_car bigint generated by default as identity
    ,car_brand string
    ,car_model string
    ,car_part string
    ,valid_from timestamp
    ,valid_to timestamp
  )

In [0]:
%sql
insert into workspace.pj_sales.tb_dim_car_gold
values (-1, 'unknown', 'unknown', 'unknown', null, null)

In [0]:
%sql
create or replace temp view vw_dim_car as(
  with dedup as (
    select
       car_brand
      ,car_model
      ,car_part
      ,row_number() over (
        partition by 
           car_brand
          ,car_model
          ,car_part
        order by 
           sale_date desc
          ,ingestion_timestamp desc
      )                                                             as order_by_date
    from workspace.pj_sales.tb_sales_silver
  )
  select
     car_brand
    ,car_model
    ,car_part
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as valid_from
    ,null                                                           as valid_to
  from dedup
  where order_by_date = 1
)

In [0]:
%sql
create or replace temp view vw_dim_car_merge as
select
  vcar.*,
  'u' as action
from vw_dim_car vcar -- Aqui o registro serve para ser atualizado
union all
select
  vcar.*,
  'i' as action
from vw_dim_car vcar -- Aqui o registro serve para ser inserido

In [0]:
%sql
merge into workspace.pj_sales.tb_dim_car_gold tcar
using vw_dim_car_merge vcarm
on  tcar.car_brand = vcarm.car_brand
and tcar.car_model = vcarm.car_model
and tcar.car_part = vcarm.car_part
and tcar.valid_to is null
and vcarm.action = 'u'
when matched and vcarm.action = 'u' then
  update set
    tcar.valid_to = vcarm.valid_from
when not matched and vcarm.action = 'i' then
  insert (
     car_brand
    ,car_model
    ,car_part
    ,valid_from
    ,valid_to
  )
  values (
     vcarm.car_brand
    ,vcarm.car_model
    ,vcarm.car_part
    ,vcarm.valid_from
    ,vcarm.valid_to
  )


In [0]:
%sql
select * from pj_sales.tb_dim_car_gold